## Using the Workforce Python Module to Automate Workforce for ArcGIS

Workforce for ArcGIS is a mobile solution that uses the power of location-based decision making for better field workforce coordination and teamwork. It is composed of a web app used by project administrators and dispatchers in the office, and a mobile app used by mobile workers on their devices. Organizations using Workforce for ArcGIS get these benefits:

- Everything you need on one device—Mobile workers can easily view and process work assignments, provide updates on work status, and inform others of their location, all from one device.

- Greater agility—Using real-time and location-based information, dispatchers can assign and prioritize fieldwork on the fly and ensure that work is assigned to the right people at the right time.

- Increased productivity—Replace time-consuming and error-prone manual workforce management processes, reduce downtime, and keep projects on schedule.

### Workforce Schema

**Note**: The following was extracted/summarized from [here](https://doc.arcgis.com/en/workforce/android-phone/help/workforce-schema.htm).

A workforce project is composed of four feature layers and four coded value domains with a predefined schema. The name of each feature layer is a combination of a moniker, describing the purpose of the feature layer, appended with the GUID of the Workforce project item. For example, the Workers layer associated with a project with GUID 5dd018fcd88c4d33814cf3da9c44061e would be named workers_5dd018fcd88c4d33814cf3da9c44061e. This guarantees uniqueness of each feature layer.

The four feature layers are as follows:

- **Workers**

  - A point feature layer that contains a record for each mobile worker who is included in the project.
  - Includes information about the mobile worker, including their contact number and job title.
  - The mobile worker's ArcGIS organizational user name is stored in the userId field.
  - The layer tracks who created and last updated each mobile worker.
  - There is a primary key-foreign key (PK-FK) relationship from OBJECTID to Assignments.workerId. Using the OBJECTID value from the Workers layer as the Assignments.workerId field value associates the mobile worker with all their assignments.
  - The layer has the following coded value domain associations:
  - The status field is associated with the Worker_Status coded value domain to track the mobile worker status.

- **Assignments**

  - A point feature layer that contains a record for each assignment.
  - Includes information about the assignment, including its status, location, and description, among others.
  - The layer tracks who created and last updated each assignment.
  - Attachments are enabled on the feature layer.
  - The layer contains foreign keys for some fields, associating values from another layer with this layer:
    - Assignments.workerId to Workers.OBJECTID.
    - Assignments.dispatcherId to Dispatchers.OBJECTID.
    - Assignments.workOrderId can be used as a foreign key to an external system, such as an asset or maintenance management system, by providing values from the other system.
  - The layer has the following coded value domain associations:
    - The status field is associated with the Assign_Status coded value domain to track the assignment status.
    - The priority field is associated with the Priority coded value domain to manage the priority of work assignments.
    - The assignmentType field is associated with the Assign_Type coded value domain to store the assignment types for the project.

- **Dispatchers**
  - A point feature layer that contains a record for each dispatcher within the project.
  - Includes information about the dispatcher, including their name and contact number.
  - The dispatcher's ArcGIS organizational user name is stored in the userId field.
  - The layer tracks who created and last updated each dispatcher.
  - There is a PK-FK relationship from OBJECTID to Assignments.dispatcherId. Using the OBJECTID value from the Dispatchers layer as the Assignments.dispatcherId field value associates the dispatcher with all the assignments they assigned.

- **Location Tracking**
  - A point feature layer that contains a record for each point location logged while location tracking is enabled.
  - The layer tracks who created and last updated each location track.
  
Additionally, a workforce project contains two webmaps:

- **Dispatcher Webmap**
  - This map is what the dispatchers using the back-office web app see
  - It shows the assignments and worker locations
  - Additional layers can be added to this map
  
- **Worker Webmap**
  - This map is what a field worker uses on their iOS or Android device
  - Additional layers can be added to this map
  
When a new project is created via the web application, a new **Group** is created. All of the layers, webmaps, and the project item itself are shared into this group.

When a new project is created via the web application, a new **Folder** is created. All of the layers, webmaps, and the project item itself are shared into this folder.

Finally, there is the actual **Workforce Project** item. This is an item on the portal that stores project meta data in json format.

```json
{
    "workerWebMapId": "<worker-map-id>",
    "dispatcherWebMapId": "<dispatcher-map-id>",
    "dispatchers": {
        "serviceItemId": "<item-id>",
        "url": "<layer-url>"
    },
    "assignments": {
        "serviceItemId": "<item-id>",
        "url": "<layer-url>"
    },
    "workers": {
        "serviceItemId": "<item-id>",
        "url": "<layer-url>"
    },
    "tracks": {
        "serviceItemId": "<item-id>",
        "url": "<layer-url>",
        "enabled": <true | false>,
        "updateInterval": 30
    },
    "version": "1.2.0",
    "groupId": "<group-id>",
    "folderId": "<folder-d>",
    "assignmentIntegrations": [
        {
            "id": "default-navigator",
            "prompt": "Navigate to Assignment",
            "urlTemplate": "arcgis-navigator://?stop=${assignment.latitude},${assignment.longitude}&stopname=${assignment.location}&callback=arcgis-workforce://&callbackprompt=Workforce"
        }
    ]
}
```


### Common tasks that can be accomplished with this module

#### Project
- Renaming a Project
- Deleting a Project

#### Workers and Dispatchers
- Adding Dispatchers and Workers to a Project
- Removing Dispatchers and Workers from a Project
- Updating Workers and Dispatchers in a Project
- Querying Workers and Dispatchers in a Project

#### Assignments
- Adding Assignments to a Project
- Removing Assignments from a Project
- Updating Assignments in a Project
- Assigning Assignments in a Project
- Querying Assignments in a Project
- Adding/Removing/Downloading Attachments

#### Tracks
- Querying Tracks (for analysis)

### Getting Started

A user must be logged on to a GIS in order to fetch a Project. The workforce functionality is available in `arcgis.apps.workforce`

In [1]:
from arcgis.gis import GIS
from arcgis.apps import workforce

gis = GIS('https://arcgis.com', 'workforce_scripts', 'esri12345')

### Accessing a project

In [2]:
item = gis.content.get("29eec3c45982458d876c5e1f2f32333d")
project = workforce.Project(item)

#### Accessing assignments in a project
There is a helper class called the `AssignmentManager`. This class can be used to search, update, and delete assignments in bulk. It is available by calling `project.assignments`.

In [3]:
# Get all of the assignments in a project
assignments = project.assignments.search()

# Let's view the first assignment
assignment = assignments[0]
print(f"Status: {assignment.status}")
print(f"Description: {assignment.description}")
print(f"Priority: {assignment.priority}")
print(f"Assigned To: {assignment.worker.name}")
print(f"Type: {assignment.assignment_type}")

# Let's update the description
assignment.update(description="You need to do an inspection here")
print("--------------------")
print(f"Updated Description: {project.assignments.search()[0].description}")

# Let's download the attachments for this assignment using the AssignmentAttachmentManager
assignment.attachments.download()

Status: assigned
Description: Do some work at the ESRI R&D Center
Priority: medium
Assigned To: Aaron Pulver
Type: Inspection
--------------------
Updated Description: You need to do an inspection here


['/Users/aaro8157/PycharmProjects/geosaurus/examples/Apps/esri_logo1.png']

#### Accessing assignment types in a project
There is a helper class called the `AssignmentTypeManager`. This class can be used to search, update, and delete assignment types in bulk. It is available by calling `project.assignment_types`.

In [4]:
# Get all of the assignment types
assignment_types = project.assignment_types.search()
for at in assignment_types:
    print(f"Type: {at.name}")
    
# Let's add a new assignment type
project.assignment_types.add(name="Repair")

# Let's confirm that it was added
print("--------------------")
assignment_types = project.assignment_types.search()
for at in assignment_types:
    print(f"Type: {at.name}")

Type: Inspection
Type: Removal
--------------------
Type: Inspection
Type: Removal
Type: Repair


#### Accessing workers in a project
There is a helper class called the `WorkerManager`. This class can be used to search, update, and delete workers in bulk. It is available by calling `project.workers`.

In [5]:
# Get all of the workers
workers = project.workers.search()
worker = workers[0]
print(f"Name: {worker.name}")
print(f"Number: {worker.contact_number}")
    
# Let's update the workers number
worker.update(contact_number="123-456-7890")
print("--------------------")
print(f"Number: {project.workers.search()[0].contact_number}")

Name: Aaron Pulver
Number: None
--------------------
Number: 123-456-7890


In [6]:
# Let's add a new worker
worker = project.workers.add(name="Demo User", user_id="demouser_nitro", contact_number="123-987-4560")

#### Accessing dispatchers in a project
There is a helper class called the `DispatcherManager`. This class can be used to search, update, and delete dispatcher in bulk. It is available by calling `project.dispatchers`.

In [7]:
# Get all of the dispatchers
dispatchers = project.dispatchers.search()
dispatcher = dispatchers[0]
print(f"Name: {dispatcher.name}")
print(f"Number: {dispatcher.contact_number}")
    
# Let's update the dispatchers number
dispatcher.update(contact_number="123-456-7890")
print("--------------------")
print(f"Number: {project.dispatchers.search()[0].contact_number}")

Name: workforce scripts
Number: 987-654-3210
--------------------
Number: 123-456-7890


#### Accessing the webmaps

In [8]:
# worker webmap
project.worker_webmap
# dispatcher webmap
project.dispatcher_webmap

### Putting it all together
Let's create a new assignment using data already in our project.

In [9]:
# Let's add a new assignment and assign it to demouser
from datetime import datetime
demouser = project.workers.get(user_id='demouser_nitro')
dispatcher = project.dispatchers.get(user_id='workforce_scripts')
repair = project.assignment_types.get(name="Repair")

# Let's use the geocoder to find the ESRI Campus
from arcgis.geocoding import geocode
geometry = geocode("ESRI, Redlands, CA", out_sr=102100)[0]['location']

new_assignment = project.assignments.add(assignment_type=repair, 
                        status="assigned",
                        assigned_date=datetime.now(),
                        worker=worker,
                        dispatcher=dispatcher,
                        location="ESRI, Redlands, CA",
                        geometry=geometry)

In [10]:
# Show the new assignment in Redlands
project.dispatcher_webmap

### Batch operations

There are several methods such as `batch_add`, `batch_update`, and `batch_delete` which will modify many objects at a time. These methods are available on the various manager classes. These methods may be useful when importing, editing, or deleting large amounts of assignments, dispatchers, workers, or tracks.

In [ ]:
# Let's create a bunch of assignments at the ESRI Campus
assignments = []
for i in range(0, 3):
    assignment = workforce.Assignment(project, 
                        assignment_type=repair, 
                        status="assigned",
                        assigned_date=datetime.now(),
                        worker=worker,
                        dispatcher=dispatcher,
                        location="ESRI, Redlands, CA",
                        geometry=geometry)
    assignments.append(assignment)
# call the batch_add method to create 3 assignments at once
project.assignments.batch_add(assignments)

[{"attributes": {"assignedDate": 1521741279588.5662, "assignmentRead": 0, "completedDate": null, "declinedComment": null, "declinedDate": null, "description": null, "dueDate": null, "inProgressDate": null, "location": "ESRI, Redlands, CA", "notes": null, "pausedDate": null, "priority": null, "status": 1, "workOrderId": null, "assignmentType": 3, "dispatcherId": 1, "workerId": 23, "OBJECTID": 46, "GlobalID": "B6CA2238-8FE8-4F1C-8FB9-0F7E464C8B1E"}, "geometry": {"x": -13046163.418234022, "y": 4036389.770988603}},
 {"attributes": {"assignedDate": 1521741279590.875, "assignmentRead": 0, "completedDate": null, "declinedComment": null, "declinedDate": null, "description": null, "dueDate": null, "inProgressDate": null, "location": "ESRI, Redlands, CA", "notes": null, "pausedDate": null, "priority": null, "status": 1, "workOrderId": null, "assignmentType": 3, "dispatcherId": 1, "workerId": 23, "OBJECTID": 47, "GlobalID": "CDDC329C-4E60-4589-8D15-2407E3C4F817"}, "geometry": {"x": -13046163.4182

### Reset the Demo Project

In [ ]:
project.assignments.batch_delete(project.assignments.search(where='assignmentType=3'))
repair_type = project.assignment_types.get(name="Repair")
if repair_type:
    project.assignment_types.batch_delete([repair_type])
project.workers.batch_delete(project.workers.search(where="userId='demouser_nitro'"))
a = project.assignments.get(object_id=1)
a.update(description="Do some work at the ESRI R&D Center")

w1 = project.workers.get(object_id=1)
w1.contact_number = None
w1.update()
d1 = project.dispatchers.get(object_id=1)
d1.update(contact_number="987-654-3210")